# [Super AI Engineer Season 6] Mini Hackathon 2 Level 2
## Parasite Eggs Classification

**Super AI Engineer Season 6 - Level 2 Hackathon**  
- Dataset: Chula-ParasiteEgg-11 + Competition Test Set
- จัดทำโดย: 600425-วิศิษฐ์

---
### Notebook Outline
1. Setup & Imports  
2. Data Loading (Parsing Filenames)
3. Data Preparation  
4. Model Training (EfficientNet-B3)  
5. OOD Threshold Calibration  
6. Test Inference & Submission

# 1. Setup & Imports
### 1.1 นำเข้าไลบรารีและกำหนดค่าพื้นฐาน

In [1]:
!pip install -q timm

import os, random, warnings, re
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from torchvision import transforms
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Class Configuration ──────────────────────────────────────────────────────
# ตรงตามโครงสร้าง Chula-ParasiteEgg-11 (Index 0-10)
CLASSES = [
    "Ascaris lumbricoides",       # 0
    "Capillaria philippinensis",  # 1
    "Enterobius vermicularis",    # 2
    "Fasciolopsis buski",         # 3
    "Hookworm egg",               # 4
    "Hymenolepis diminuta",       # 5
    "Hymenolepis nana",           # 6
    "Opisthorchis viverrine",     # 7
    "Paragonimus spp",            # 8
    "Taenia spp. egg",            # 9
    "Trichuris trichiura",        # 10
]
NUM_CLASSES = len(CLASSES)
CLASS2IDX   = {c: i for i, c in enumerate(CLASSES)}
IDX2CLASS   = {i: c for i, c in enumerate(CLASSES)}

# ── Path Configuration ──────────────────────────────────────────────────────
# สำหรับ Kaggle: ต้องเพิ่ม dataset "macharning/chula-parasite-dataset"
TRAIN_DIR  = Path('/kaggle/input/datasets/macharning/chula-parasite-dataset/Chula-ParasiteEgg-11/Chula-ParasiteEgg-11/Chula-ParasiteEgg-11/data')
TEST_DIR   = Path('/kaggle/input/competitions/super-ai-engineer-season-6-parasite-eggs/test_set/test')
SAMPLE_SUB = Path('/kaggle/input/competitions/super-ai-engineer-season-6-parasite-eggs/sample_submission.csv')
WORK_DIR   = Path('/kaggle/working')

print(f'Device: {DEVICE}')

Device: cuda


# 2. Data Loading
### 2.1 อ่านไฟล์และแยก Class จากชื่อไฟล์ (Filenames)

In [2]:
# เนื่องจากใน dataset Chula-ParasiteEgg-11 ไฟล์ทั้งหมดอยู่ใน folder 'data' 
# และใช้ชื่อไฟล์เป็นตัวบอก class เช่น 'Ascaris lumbricoides_0001.jpg'

all_paths, all_labels = [], []

if TRAIN_DIR.exists():
    img_files = list(TRAIN_DIR.glob('*.jpg')) + list(TRAIN_DIR.glob('*.png'))
    print(f'Found {len(img_files):,} images in {TRAIN_DIR}')
    
    for p in img_files:
        # แยกชื่อ class จาก filename (ตัด _xxxx.jpg ออก)
        fname = p.name
        matched_class = None
        for c in CLASSES:
            if fname.startswith(c):
                matched_class = c
                break
        
        if matched_class:
            all_paths.append(p)
            all_labels.append(CLASS2IDX[matched_class])
else:
    print(f'WARNING: TRAIN_DIR not found at {TRAIN_DIR}')

# สรุปจำนวนข้อมูลต่อ class
if all_labels:
    counts = pd.Series(all_labels).map(IDX2CLASS).value_counts().sort_index()
    print('\nTraining Data Distribution:')
    print(counts)

Found 11,000 images in /kaggle/input/datasets/macharning/chula-parasite-dataset/Chula-ParasiteEgg-11/Chula-ParasiteEgg-11/Chula-ParasiteEgg-11/data

Training Data Distribution:
Ascaris lumbricoides         1000
Capillaria philippinensis    1000
Enterobius vermicularis      1000
Fasciolopsis buski           1000
Hookworm egg                 1000
Hymenolepis diminuta         1000
Hymenolepis nana             1000
Opisthorchis viverrine       1000
Paragonimus spp              1000
Taenia spp. egg              1000
Trichuris trichiura          1000
Name: count, dtype: int64


# 3. Data Preparation
### 3.1 Dataset และ Transforms

In [3]:
IMG_SIZE = 224

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ParasiteDataset(Dataset):
    def __init__(self, paths, labels=None, tf=None):
        self.paths  = paths
        self.labels = labels
        self.tf     = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.tf: img = self.tf(img)
        if self.labels is not None: return img, self.labels[idx]
        return img

# Train/Val Split
tr_paths, vl_paths, tr_lbls, vl_lbls = train_test_split(
    all_paths, all_labels, test_size=0.15, stratify=all_labels, random_state=SEED
)

tr_loader = DataLoader(ParasiteDataset(tr_paths, tr_lbls, train_tf), batch_size=32, shuffle=True)
vl_loader = DataLoader(ParasiteDataset(vl_paths, vl_lbls, val_tf), batch_size=64)

# 4. Model Training
### 4.1 Fine-tune EfficientNet-B3

In [4]:
model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

EPOCHS = 15
best_f1 = 0
for epoch in range(1, EPOCHS+1):
    model.train()
    for imgs, lbls in tr_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), lbls)
        loss.backward(); optimizer.step()
    
    # Validation
    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for imgs, lbls in vl_loader:
            out = model(imgs.to(DEVICE))
            preds.extend(out.argmax(1).cpu().numpy())
            gts.extend(lbls.numpy())
    
    f1 = f1_score(gts, preds, average='macro')
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), 'best_model.pth')
    print(f'Epoch {epoch:02d} | Val F1: {f1:.4f}')

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Epoch 01 | Val F1: 0.8950
Epoch 02 | Val F1: 0.9547
Epoch 03 | Val F1: 0.9684
Epoch 04 | Val F1: 0.9739
Epoch 05 | Val F1: 0.9776
Epoch 06 | Val F1: 0.9825
Epoch 07 | Val F1: 0.9854
Epoch 08 | Val F1: 0.9861
Epoch 09 | Val F1: 0.9885
Epoch 10 | Val F1: 0.9867
Epoch 11 | Val F1: 0.9824
Epoch 12 | Val F1: 0.9867
Epoch 13 | Val F1: 0.9903
Epoch 14 | Val F1: 0.9909
Epoch 15 | Val F1: 0.9903


# 5. OOD Threshold Calibration
### 5.1 ตั้งค่า Threshold สำหรับ Class ที่ไม่รู้จัก (-1)

In [5]:
# โหลดโมเดลที่ดีที่สุด
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# คำนวณ Softmax Confidence บน Val set
val_probs = []
with torch.no_grad():
    for imgs, _ in vl_loader:
        out = F.softmax(model(imgs.to(DEVICE)), dim=1)
        val_probs.extend(out.cpu().numpy())

val_probs = np.array(val_probs)
val_confs = val_probs.max(axis=1)

# เราจะใช้ threshold เบื้องต้นที่ 0.50 
# (ถ้ามั่นใจน้อยกว่า 50% ให้ตอบ -1 เพราะอาจเป็น background หรือ unknown)
THRESHOLD = 0.50
print(f'Using Confidence Threshold: {THRESHOLD}')

Using Confidence Threshold: 0.5


# 6. Test Inference & Submission
### 6.1 ทำนายผลบน Test Set

In [6]:
test_files = sorted(list(TEST_DIR.glob('*.png')))
test_loader = DataLoader(ParasiteDataset(test_files, tf=val_tf), batch_size=64)

test_preds, test_confs = [], []
with torch.no_grad():
    for imgs in test_loader:
        out = F.softmax(model(imgs.to(DEVICE)), dim=1)
        confs, idxs = out.max(1)
        test_preds.extend(idxs.cpu().numpy())
        test_confs.extend(confs.cpu().numpy())

# Apply Thresholding
final_labels = []
for p, c in zip(test_preds, test_confs):
    if c >= THRESHOLD:
        final_labels.append(int(p))
    else:
        final_labels.append(-1)

# สร้าง Submission CSV
sub = pd.read_csv(SAMPLE_SUB)
# Mapping ผลลัพธ์กลับเข้าตาราง (ตรวจสอบชื่อไฟล์ให้ตรงกัน)
test_results = {f.name: lbl for f, lbl in zip(test_files, final_labels)}
sub['label'] = sub['filename'].map(test_results)

sub.to_csv('submission_PE.csv', index=False)
print('Submission saved to submission_PE.csv')
print(sub.head())

Submission saved to submission_PE.csv
     filename  label
0  000000.png      6
1  000001.png      6
2  000002.png      6
3  000003.png     -1
4  000004.png     -1
